# DarkSpot — аренда коммерции с ЦИАНа

Парсим cian.ru, смотрим на цены, метро и типы помещений по Москве. Парсинг статический — cloudscraper + BeautifulSoup.

## Как это работает

ЦИАН рендерит страницы через JS, но начальные данные лежат прямо в HTML — в script-теге в виде JSON.
Поэтому можно обойтись обычным HTTP-запросом без браузера.

Инструменты: cloudscraper (обход Cloudflare), BeautifulSoup (парсинг HTML), json (разбор данных), ThreadPoolExecutor (параллельный сбор по типам объектов).

Типы объектов: офисы, торговые площади, свободное назначение, склады.

Поля датасета:
- price_month — итоговая цена аренды в месяц, руб
- price_sqm_month — цена за м² в месяц
- area_sqm — площадь помещения, м²
- okrug / district — округ и район Москвы
- metro_station / metro_walk_min — ближайшее метро и расстояние до него пешком
- floor / floors_total — этаж и этажность здания
- building_class — класс здания (A, B+, B, C)
- building_type — тип здания (БЦ, ТЦ, жилой дом и др.)
- renovation — вид ремонта
- ceiling_height_m — высота потолков из описания, м


### Часть 1. Окружение

In [1]:
!pip install -q playwright plotly dash pandas numpy
!playwright install chromium --with-deps
!pip install -q cloudscraper beautifulsoup4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.5/47.5 MB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 44.9 MB/s eta 0:00:00
Installing dependencies...
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [95.6 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,646 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:11 http://archive.ubuntu.co

In [2]:
import os, re, json, time, random, threading, warnings
from abc import ABC, abstractmethod
from typing import List, Dict, Optional
import cloudscraper
from bs4 import BeautifulSoup

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)

### Часть 2. ООП

Базовый класс задаёт интерфейс и кеширование. Конкретный парсер реализует `fetch()` и `parse()`.

In [16]:
class BaseParser(ABC):
    # базовый класс для всех парсеров, от него наследуются конкретные парсеры

    def __init__(self, cache_dir="darkspot_cache", use_cache=True):
        self.cache_dir = cache_dir
        self.use_cache = use_cache
        self.df = None
        self.source_name = self.__class__.__name__

    @property
    def cache_path(self):
        # путь к файлу с кэшом
        return os.path.join(self.cache_dir, f"{self.source_name}_raw.csv")

    @abstractmethod
    def fetch(self): ...

    @abstractmethod
    def parse(self, raw): ...

    def collect(self):
        # если кэш есть, грузим его и не парсим заново
        if self.use_cache and os.path.exists(self.cache_path):
            print(f"[{self.source_name}] берем из кэша: {self.cache_path}")
            self.df = pd.read_csv(self.cache_path)
            return self.df

        print(f"[{self.source_name}] запускаем парсинг...")
        raw = self.fetch()
        self.df = self.parse(raw)

        os.makedirs(self.cache_dir, exist_ok=True)
        self.df.to_csv(self.cache_path, index=False)
        print(f"готово, {len(self.df)} строк сохранено в {self.cache_path}")
        return self.df


print("все норм")

все норм


### Часть 3. CianCommercialParser


ЦИАН использует Cloudflare WAF и JavaScript-рендеринг. `cloudscraper` обходит защиту на уровне  
HTTP-заголовков, имитируя браузер Chrome. Данные извлекаются из script-тега с JSON внутри HTML.

Playwright на ЦИАНе не заработал — сайт детектит автоматизированный браузер и отдает капчу, поэтому перешли на cloudscraper

In [17]:
class CianCommercialParser(BaseParser):
    # конкретный парсер для коммерческой аренды на циане, наследует BaseParser

    # id типов объектов в апи циана
    OFFER_TYPE_IDS = {
        "offices":               "1",
        "shoppingAreas":         "2",
        "warehouse":             "3",
        "freeAppointmentObject": "5",
    }

    # человекочитаемые названия типов
    OFFER_TYPE_NAMES = {
        "offices":               "Офис",
        "shoppingAreas":         "Торговая площадь",
        "warehouse":             "Склад",
        "freeAppointmentObject": "Свободное назначение",
    }

    # маппинг ремонта из апи в русский
    DEC_MAP = {
        "design":   "дизайнерский",
        "euro":     "евроремонт",
        "fine":     "евроремонт",
        "cosmetic": "косметический",
        "no":       "без отделки",
    }

    # маппинг классов зданий
    CLS_MAP = {
        "a": "A", "b_plus": "B+", "bPlus": "B+",
        "b": "B", "c": "C", "d": "C",
    }

    # маппинг типов зданий
    BUILDING_TYPE_MAP = {
        "businessCenter":         "Бизнес-центр",
        "shoppingCenter":         "Торговый центр",
        "shoppingEntertainment":  "ТРЦ",
        "warehouse":              "Склад",
        "manufacturingFacility":  "Производство",
        "residentialHouse":       "Жилой дом",
        "standalone":             "Отдельное здание",
        "administrativeBuilding": "Административное здание",
    }

    def __init__(self, offer_types=None, max_pages=5, max_workers=4, **kwargs):
        super().__init__(**kwargs)
        self.offer_types = offer_types or ["shoppingAreas"]
        self.max_pages = max_pages
        self.max_workers = max_workers
        self._lock = __import__("threading").Lock()

    def _make_scraper(self):
        # создаем скрапер, притворяется chrome на маке
        return cloudscraper.create_scraper(
            browser={"browser": "chrome", "platform": "darwin", "mobile": False}
        )

    def fetch(self):
        # запускаем парсинг всех типов объектов параллельно
        from concurrent.futures import ThreadPoolExecutor, as_completed

        all_raw = []
        futures = {}

        with ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            for offer_type in self.offer_types:
                future = executor.submit(self._fetch_type, offer_type)
                futures[future] = offer_type

            for future in as_completed(futures):
                offer_type = futures[future]
                try:
                    rows = future.result()
                    all_raw.extend(rows)
                except Exception as e:
                    print(f"[{self.OFFER_TYPE_NAMES.get(offer_type)}] ошибка: {e}")

        return all_raw

    def _fetch_type(self, offer_type):
        # парсим страницы одного типа объектов
        scraper = self._make_scraper()
        type_id = self.OFFER_TYPE_IDS.get(offer_type, "2")
        name = self.OFFER_TYPE_NAMES.get(offer_type, offer_type)
        type_raw = []

        for p_num in range(1, self.max_pages + 1):
            url = (
                f"https://www.cian.ru/cat.php?deal_type=rent&engine_version=2"
                f"&offer_type=offices&office_type%5B0%5D={type_id}&p={p_num}&region=1"
            )
            try:
                resp = scraper.get(url, timeout=20)

                if resp.status_code == 403:
                    with self._lock:
                        print(f"[{name}] стр.{p_num}: заблокировали (403)")
                    break
                if resp.status_code != 200:
                    with self._lock:
                        print(f"[{name}] стр.{p_num}: ошибка {resp.status_code}")
                    break

                rows = self._extract(resp.text, offer_type)

                if not rows:
                    with self._lock:
                        print(f"[{name}] стр.{p_num}: данных нет, заканчиваем")
                    break

                type_raw.extend(rows)
                with self._lock:
                    print(f"[{name}] стр.{p_num}: +{len(rows)} (итого {len(type_raw)})")

                # пауза чтобы не словить бан
                if p_num % 10 == 0:
                    time.sleep(random.uniform(8.0, 12.0))
                else:
                    time.sleep(random.uniform(0.7, 1.2))

            except Exception as e:
                with self._lock:
                    print(f"[{name}] стр.{p_num}: {e}")
                break

        return type_raw

    def _extract(self, html, offer_type):
        # вытаскиваем массив офферов из script-тега внутри html
        soup = BeautifulSoup(html, "html.parser")

        script_text = None
        for script in soup.find_all("script"):
            t = script.string or ""
            if "bargainTerms" in t and len(t) > 100_000:
                script_text = t
                break

        if not script_text:
            return []

        # ищем массив offers и парсим его как json
        key = '"offers":'
        idx = script_text.find(key)
        while idx >= 0:
            arr_start = script_text.index("[", idx)
            depth, i = 0, arr_start
            while i < len(script_text):
                if script_text[i] == "[":
                    depth += 1
                elif script_text[i] == "]":
                    depth -= 1
                    if depth == 0:
                        candidate = script_text[arr_start: i + 1]
                        if '"bargainTerms"' in candidate:
                            try:
                                offers = json.loads(candidate)
                                return [r for o in offers if o
                                        for r in [self._row(o, offer_type)] if r]
                            except Exception:
                                pass
                        break
                i += 1
            idx = script_text.find(key, idx + 1)

        return []

    def _row(self, o, offer_type):
        # собираем одну строку датасета из сырого объекта оффера
        try:
            bt = o.get("bargainTerms", {}) or {}
            price_rur = float(
                o.get("priceTotalPerMonthRur") or
                bt.get("priceRur") or
                bt.get("price") or
                0
            )
            area = float(o.get("totalArea") or o.get("minArea") or 0)
            if price_rur <= 0 or area <= 0:
                return None

            price_sqm = round(price_rur / area, 1)
            if price_sqm < 100:
                return None

            # разбираем адрес
            geo = o.get("geo", {}) or {}
            district = okrug = street = house = ""
            for a in geo.get("address", []) or []:
                t, n = a.get("type", ""), a.get("name", "")
                if t == "raion":    district = n
                elif t == "okrug":  okrug = n
                elif t == "street": street = n
                elif t == "house":  house = n

            address = ", ".join(filter(None, [street, house]))

            # берем только метро пешком
            ug = geo.get("undergrounds") or []
            walk_metros = [u for u in ug if u.get("transportType") == "walk"]
            metro = walk_metros[0].get("name") if walk_metros else None
            metro_min = walk_metros[0].get("time") if walk_metros else None

            bld = o.get("building", {}) or {}
            bld_cls = self.CLS_MAP.get((bld.get("classType") or "").lower(), None)
            bld_type = self.BUILDING_TYPE_MAP.get(bld.get("type") or "", None)
            renovation = self.DEC_MAP.get(o.get("decoration") or "", None)

            floor = int(o["floorNumber"]) if o.get("floorNumber") is not None else None
            f_total = int(bld["floorsCount"]) if bld.get("floorsCount") is not None else None

            # высота потолков из текста описания
            desc = o.get("description", "") or ""
            ceil_match = re.search(r'высота потолков[:\s]+(\d+[.,]\d+)', desc, re.IGNORECASE)
            ceiling = float(ceil_match.group(1).replace(",", ".")) if ceil_match else None

            return {
                "cian_id":          int(o.get("id", 0)),
                "url":              f"https://www.cian.ru/rent/commercial/{o.get('id', 0)}/",
                "object_type":      self.OFFER_TYPE_NAMES.get(offer_type, offer_type),
                "price_month":      int(price_rur),
                "area_sqm":         area,
                "price_sqm_month":  price_sqm,
                "address":          address or None,
                "district":         district or None,
                "okrug":            okrug or None,
                "metro_station":    metro,
                "metro_walk_min":   metro_min,
                "floor":            floor,
                "floors_total":     f_total,
                "building_class":   bld_cls,
                "building_type":    bld_type,
                "renovation":       renovation,
                "has_furniture":    o.get("hasFurniture"),
                "has_ac":           None,
                "has_ventilation":  None,
                "ceiling_height_m": ceiling,
                "photos_count":     len(o.get("photos") or []),
                "source":           "cloudscraper",
            }
        except Exception:
            return None

    def parse(self, raw):
        # превращаем список словарей в датафрейм и чистим мусор
        if not raw:
            raise ValueError("нет данных")
        df = pd.DataFrame(raw)
        df = df[(df["price_month"] > 0) & (df["area_sqm"] > 0)]
        return df.reset_index(drop=True)


print("CianCommercialParser готов")

CianCommercialParser готов


### Часть 4. Сбор сырых данных

Запускаем парсер. Результат сохраняется в `darkspot_cache/CianCommercialParser_raw.csv`.  


In [24]:
parser = CianCommercialParser(
    offer_types=["shoppingAreas", "offices", "freeAppointmentObject", "warehouse"],
    max_pages=100,
    cache_dir="darkspot_cache",
    use_cache=False,
)

df_raw = parser.collect()

print(f"\n{'─'*55}")
print(f"  Сырых записей:  {len(df_raw):,}")
print(f"  Типов объектов: {df_raw['object_type'].value_counts().to_dict()}")
print(f"{'─'*55}")
df_raw.head(10)

[CianCommercialParser] запускаем парсинг...
[Склад] стр.1: +28 (итого 28)
[Офис] стр.1: +28 (итого 28)
[Свободное назначение] стр.1: +28 (итого 28)
[Торговая площадь] стр.1: +28 (итого 28)
[Офис] стр.2: +28 (итого 56)
[Склад] стр.2: +28 (итого 56)
[Торговая площадь] стр.2: +28 (итого 56)
[Свободное назначение] стр.2: +28 (итого 56)
[Склад] стр.3: +28 (итого 84)
[Офис] стр.3: +28 (итого 84)
[Торговая площадь] стр.3: +28 (итого 84)
[Свободное назначение] стр.3: +28 (итого 84)
[Свободное назначение] стр.4: +28 (итого 112)
[Торговая площадь] стр.4: +28 (итого 112)
[Офис] стр.4: +28 (итого 112)
[Склад] стр.4: +28 (итого 112)
[Свободное назначение] стр.5: +28 (итого 140)
[Торговая площадь] стр.5: +28 (итого 140)
[Офис] стр.5: +28 (итого 140)
[Склад] стр.5: +28 (итого 140)
[Свободное назначение] стр.6: +28 (итого 168)
[Торговая площадь] стр.6: +28 (итого 168)
[Офис] стр.6: +28 (итого 168)
[Склад] стр.6: +28 (итого 168)
[Свободное назначение] стр.7: +28 (итого 196)
[Торговая площадь] стр.7: +2

,cian_id,url,object_type,price_month,area_sqm,price_sqm_month,address,district,okrug,metro_station,metro_walk_min,floor,floors_total,building_class,building_type,renovation,has_furniture,has_ac,has_ventilation,ceiling_height_m,photos_count,source
0,326829055,https://www.cian.ru/rent/commercial/326829055/,Свободное назначение,3600000,4000.0,900.0,"Варшавское, 170Г",Чертаново Южное,ЮАО,Аннино,9.0,1,5,None,None,None,None,None,None,NaN,5,cloudscraper
1,328558674,https://www.cian.ru/rent/commercial/328558674/,Свободное назначение,480000,400.0,1200.0,"Варшавское, 170Г",Чертаново Южное,ЮАО,Аннино,9.0,1,5,None,None,None,None,None,None,NaN,5,cloudscraper
2,327533082,https://www.cian.ru/rent/commercial/327533082/,Свободное назначение,160308,73.0,2196.0,"4922-й, 4с4",Старое Крюково,ЗелАО,None,NaN,1,4,B,None,None,False,None,None,4.5,11,cloudscraper
3,306607073,https://www.cian.ru/rent/commercial/306607073/,Свободное назначение,7764250,1433.4,5416.7,"Рочдельская, 15С13",Пресненский,ЦАО,Краснопресненская,15.0,1,4,None,Бизнес-центр,None,None,None,None,NaN,33,cloudscraper
4,329734088,https://www.cian.ru/rent/commercial/329734088/,Свободное назначение,865275,17.6,49163.4,"Огородный, 4к2",Бутырский,СВАО,Бутырская,4.0,2,56,None,None,None,None,None,None,NaN,25,cloudscraper
5,329037951,https://www.cian.ru/rent/commercial/329037951/,Свободное назначение,307207,162.4,1891.7,"Смольная, 14",Головинский,САО,Речной вокзал,16.0,1,15,B,Бизнес-центр,None,None,None,None,NaN,19,cloudscraper
6,323315566,https://www.cian.ru/rent/commercial/323315566/,Свободное назначение,1441282,2042.0,705.8,82,None,ТАО (Троицкий),None,NaN,1,1,None,None,None,None,None,None,NaN,21,cloudscraper
7,322491104,https://www.cian.ru/rent/commercial/322491104/,Свободное назначение,378015,213.9,1767.3,"2-й Павелецкий, 5С1",Даниловский,ЮАО,None,NaN,3,8,B,Бизнес-центр,None,None,None,None,NaN,21,cloudscraper
8,329734077,https://www.cian.ru/rent/commercial/329734077/,Свободное назначение,1046633,19.8,52860.3,"Огородный, 4к2",Бутырский,СВАО,Бутырская,4.0,2,56,None,None,None,None,None,None,NaN,25,cloudscraper
9,308761689,https://www.cian.ru/rent/commercial/308761689/,Свободное назначение,800000,283.9,2817.9,"Верхняя Масловка, 20С1",Савеловский,САО,Петровский Парк,7.0,-1,8,None,None,None,None,None,None,NaN,34,cloudscraper


In [25]:
# Сводка по сырым данным
print("Типы и диапазоны значений:")
df_raw.info()
print()
print(df_raw[["price_month","area_sqm","price_sqm_month","metro_walk_min"]].describe().round(1))

Типы и диапазоны значений:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11147 entries, 0 to 11146
Data columns (total 22 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   cian_id           11147 non-null  int64  
 1   url               11147 non-null  object 
 2   object_type       11147 non-null  object 
 3   price_month       11147 non-null  int64  
 4   area_sqm          11147 non-null  float64
 5   price_sqm_month   11147 non-null  float64
 6   address           10795 non-null  object 
 7   district          9600 non-null   object 
 8   okrug             11132 non-null  object 
 9   metro_station     8801 non-null   object 
 10  metro_walk_min    8801 non-null   float64
 11  floor             11147 non-null  int64  
 12  floors_total      11147 non-null  int64  
 13  building_class    3817 non-null   object 
 14  building_type     4938 non-null   object 
 15  renovation        0 non-null      object 
 16  has_furniture


### Часть 5. Обработка данных

Чистим сырые данные: убираем дубли, выбросы, добавляем производные признаки.

In [26]:
df = df_raw.copy()

# дедупликация по cian_id
print(f"исходных записей: {len(df)}")

df_id  = df[df["cian_id"] > 0].drop_duplicates(subset=["cian_id"], keep="first")
df_nid = df[df["cian_id"] == 0].drop_duplicates(subset=["price_month", "area_sqm"], keep="first")
df = pd.concat([df_id, df_nid], ignore_index=True)
print(f"после дедупликации по id: {len(df)}")

# убираем полные дубли по содержимому (без технических полей)
ignore_cols = ["cian_id", "url", "source"]
content_cols = [c for c in df.columns if c not in ignore_cols]
df = df.drop_duplicates(subset=content_cols, keep="first")
print(f"после дедупликации по содержимому: {len(df)}")

# убираем выбросы по цене за м² — режем 1% снизу и сверху
q_low  = df["price_sqm_month"].quantile(0.01)
q_high = df["price_sqm_month"].quantile(0.99)
df = df[df["price_sqm_month"].between(q_low, q_high)]
print(f"после фильтрации выбросов: {len(df)}, диапазон цен: {q_low:.0f} - {q_high:.0f} руб/м²/мес")

# добавляем категориальные признаки для анализа
df["price_tier_per_meter"] = pd.cut(
    df["price_sqm_month"],
    bins=[0, 5_000, 15_000, 40_000, 999_999],
    labels=["эконом (<5k)", "средний (5-15k)", "премиум (15-40k)", "люкс (>40k)"]
)

df["area_tier"] = pd.cut(
    df["area_sqm"],
    bins=[0, 30, 80, 200, 9999],
    labels=["малый (<30 м²)", "средний (30-80)", "большой (80-200)", "крупный (>200)"]
)

df["metro_tier"] = pd.cut(
    df["metro_walk_min"],
    bins=[-1, 5, 10, 20, 999],
    labels=["до 5 мин", "6-10 мин", "11-20 мин", ">20 мин"]
)

df = df.reset_index(drop=True)

print(f"итого: {len(df)} записей, {len(df.columns)} признаков")
print(df[["object_type", "price_month", "area_sqm", "price_sqm_month",
          "price_tier_per_meter", "area_tier", "metro_tier"]].head(8))

исходных записей: 11147
после дедупликации по id: 4995
после дедупликации по содержимому: 4913
после фильтрации выбросов: 4813, диапазон цен: 730 - 20459 руб/м²/мес
итого: 4813 записей, 25 признаков
            object_type  price_month  area_sqm  price_sqm_month  \
0  Свободное назначение      3600000    4000.0            900.0   
1  Свободное назначение       480000     400.0           1200.0   
2  Свободное назначение       160308      73.0           2196.0   
3  Свободное назначение      7764250    1433.4           5416.7   
4  Свободное назначение       307207     162.4           1891.7   
5  Свободное назначение       378015     213.9           1767.3   
6  Свободное назначение       800000     283.9           2817.9   
7  Свободное назначение      2083334     500.0           4166.7   

  price_tier_per_meter         area_tier metro_tier  
0         эконом (<5k)    крупный (>200)   6-10 мин  
1         эконом (<5k)    крупный (>200)   6-10 мин  
2         эконом (<5k)   средний (3

In [27]:
# Итоговый DataFrame
print(f"Финальный датасет: {len(df)} записей × {len(df.columns)} признаков")
print()
print("Распределение по ценовым категориям:")
print(df["price_tier_per_meter"].value_counts().sort_index())
print()
print("Распределение по площади:")
print(df["area_tier"].value_counts().sort_index())

Финальный датасет: 4813 записей × 25 признаков

Распределение по ценовым категориям:
price_tier_per_meter
эконом (<5k)        3896
средний (5-15k)      879
премиум (15-40k)      38
люкс (>40k)            0
Name: count, dtype: int64

Распределение по площади:
area_tier
малый (<30 м²)       472
средний (30-80)      970
большой (80-200)    1185
крупный (>200)      2124
Name: count, dtype: int64


### Часть 6. Анализ данных

Анализируем собранные данные по четырём направлениям:

1. **Распределение цен** - как распределены цены аренды, где медиана рынка
2. **Цена vs площадь** - зависимость ставки от размера помещения
3. **Влияние метро** - как близость к метро влияет на цену аренды
4. **Структура рынка** - соотношение ценовых сегментов


## 6.1. Распределение цен

In [28]:
fig = px.histogram(
    df, x="price_sqm_month",
    nbins=40,
    title="Распределение цены аренды торговых площадей (руб/м²/мес)",
    labels={"price_sqm_month": "Цена, руб/м²/мес", "count": "Объявлений"},
    color_discrete_sequence=["#1d4ed8"],
)
fig.add_vline(x=df["price_sqm_month"].median(), line_dash="dash", line_color="red",
              annotation_text=f"Медиана: {df['price_sqm_month'].median():,.0f}",
              annotation_position="top right")
fig.update_layout(plot_bgcolor="white", height=400)
fig.show()

## 6.2. Цена vs площадь

In [29]:
fig = px.scatter(
    df,
    x="area_sqm", y="price_sqm_month",
    size="price_month", size_max=20,
    color="price_tier_per_meter",
    hover_data=["metro_station","metro_walk_min","floor"],
    title="Площадь vs Цена за м² (размер точки = общая цена аренды)",
    labels={"area_sqm": "Площадь, м²", "price_sqm_month": "Цена, руб/м²/мес",
            "price_tier_per_meter": "Категория"},
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.update_layout(plot_bgcolor="white", height=460)
fig.show()

## 6.3. Цена vs расстояние до метро

In [30]:
metro_stats = (
    df[df["metro_walk_min"] > 0]
    .groupby("metro_tier", observed=True)["price_sqm_month"]
    .agg(["median","count"])
    .reset_index()
    .rename(columns={"median":"Медиана цены","count":"Объявлений"})
)

fig = px.bar(
    metro_stats,
    x="metro_tier", y="Медиана цены",
    text="Медиана цены",
    title="Медианная цена аренды vs удалённость от метро",
    labels={"metro_tier": "Расстояние до метро", "Медиана цены": "руб/м²/мес"},
    color="Медиана цены",
    color_continuous_scale="Blues",
)
fig.update_traces(texttemplate="%{text:,.0f}", textposition="outside")
fig.update_layout(plot_bgcolor="white", height=390, coloraxis_showscale=False)
fig.show()

## 6.4. Распределение по ценовым категориям

In [31]:
tier_stats = df["price_tier_per_meter"].value_counts().sort_index().reset_index()
tier_stats.columns = ["Категория","Кол-во"]

fig = px.pie(
    tier_stats,
    names="Категория", values="Кол-во",
    title="Структура рынка торговой аренды по ценовым категориям",
    hole=0.4,
    color_discrete_sequence=px.colors.qualitative.Pastel,
)
fig.update_layout(height=400)
fig.show()

## 6.5. Топ-10 объявлений по соотношению цена/площадь

In [32]:
top = (
    df.sort_values("price_sqm_month")
    .head(10)
    [["url", "price_month", "area_sqm", "price_sqm_month",
      "metro_station", "metro_walk_min", "floor", "floors_total"]]
    .reset_index(drop=True)
)
top.index += 1
display(top)

,url,price_month,area_sqm,price_sqm_month,metro_station,metro_walk_min,floor,floors_total
1,https://www.cian.ru/rent/commercial/326141208/,590000,807.0,731.1,None,NaN,3,3
2,https://www.cian.ru/rent/commercial/329875556/,147600,200.0,738.0,Марьина Роща,15.0,1,1
3,https://www.cian.ru/rent/commercial/330183885/,74000,100.0,740.0,Окружная,4.0,-1,5
4,https://www.cian.ru/rent/commercial/327834778/,650000,878.0,740.3,Озёрная,20.0,1,5
5,https://www.cian.ru/rent/commercial/324542266/,120000,161.6,742.6,None,NaN,1,15
6,https://www.cian.ru/rent/commercial/325278524/,74700,100.0,747.0,Ростокино,14.0,1,1
7,https://www.cian.ru/rent/commercial/329906270/,225000,300.0,750.0,None,NaN,1,1
8,https://www.cian.ru/rent/commercial/311083232/,59250,79.0,750.0,Серпуховская,16.0,2,2
9,https://www.cian.ru/rent/commercial/326830419/,240000,320.0,750.0,Андроновка,9.0,-1,4
10,https://www.cian.ru/rent/commercial/327464174/,1515750,2021.0,750.0,Текстильщики,16.0,1,1


### Часть 7. Визуализация

Набор интерактивных графиков Plotly и Dash для изучения рынка коммерческой аренды в разрезе  
округов, типов объектов и ценовых категорий.


In [33]:
import plotly.express as px

# медиана цены по округам
bar_df = df.groupby("okrug")["price_sqm_month"].median().sort_values(ascending=False).reset_index()
fig1 = px.bar(
    bar_df, x="okrug", y="price_sqm_month",
    title="медиана цены аренды по округам, руб/м²/мес",
    color="price_sqm_month", color_continuous_scale="Blues", height=400,
)
fig1.update_layout(plot_bgcolor="white", coloraxis_showscale=False)
fig1.show()

# доля типов объектов
fig2 = px.pie(
    df["object_type"].value_counts().reset_index(),
    names="object_type", values="count",
    title="структура по типам объектов",
    hole=0.4, color_discrete_sequence=px.colors.qualitative.Pastel, height=380,
)
fig2.show()

# площадь vs цена за м²
fig3 = px.scatter(
    df, x="area_sqm", y="price_sqm_month",
    color="object_type", size="price_month", size_max=18, opacity=0.7,
    hover_data=["okrug", "district", "metro_station", "metro_walk_min", "floor", "address"],
    title="площадь vs цена за м²",
    color_discrete_sequence=px.colors.qualitative.Set2, height=450,
)
fig3.update_layout(plot_bgcolor="white")
fig3.show()

# разброс цен по типам
fig4 = px.box(
    df, x="object_type", y="price_sqm_month", color="object_type",
    title="разброс цен по типам объектов",
    color_discrete_sequence=px.colors.qualitative.Set2, height=400,
)
fig4.update_layout(plot_bgcolor="white", showlegend=False)
fig4.show()

# цена vs расстояние до метро
metro_df = (
    df[df["metro_walk_min"].notna()]
    .groupby("metro_tier", observed=True)["price_sqm_month"]
    .median().reset_index()
)
fig5 = px.bar(
    metro_df, x="metro_tier", y="price_sqm_month",
    title="цена аренды vs расстояние до метро",
    color="price_sqm_month", color_continuous_scale="Oranges",
    height=380, text_auto=".0f",
)
fig5.update_layout(plot_bgcolor="white", coloraxis_showscale=False)
fig5.show()

# топ-20 самых дорогих объектов
top_df = (
    df.sort_values("price_sqm_month", ascending=False)
    .head(20)
    [["object_type", "okrug", "district", "address",
      "area_sqm", "price_month", "price_sqm_month", "metro_station", "metro_walk_min"]]
    .reset_index(drop=True)
)
top_df.index += 1
display(top_df)

,object_type,okrug,district,address,area_sqm,price_month,price_sqm_month,metro_station,metro_walk_min
1,Торговая площадь,САО,Беговой,"Ленинградский, 33К3",15.3,310000,20261.4,Динамо,3.0
2,Торговая площадь,ЦАО,Мещанский,"Сретенка, 1С1",100.0,2000000,20000.0,Чистые пруды,7.0
3,Торговая площадь,ЦАО,Пресненский,"Большая Садовая, 6С2",27.7,550000,19855.6,Маяковская,5.0
4,Торговая площадь,ЦАО,Пресненский,"Большая Садовая, 6С2",27.7,550000,19855.6,Белорусская,17.0
5,Торговая площадь,ЦАО,Басманный,"Бауманская, 33/2С3",17.7,350000,19774.0,Бауманская,1.0
6,Свободное назначение,ЦАО,Пресненский,"Большая Садовая, 6С2",27.0,520000,19259.3,Маяковская,5.0
7,Свободное назначение,ЦАО,Мещанский,"Большой Сухаревский, 23С2",23.4,429999,18376.0,Сухаревская,2.0
8,Торговая площадь,ЦАО,Басманный,"Мясницкая, 8/2",75.3,1350000,17928.3,Лубянка,3.0
9,Торговая площадь,ЮЗАО,Академический,"Профсоюзная, 1/24",24.0,420000,17500.0,Академическая,1.0
10,Свободное назначение,ЦАО,Пресненский,"Большая Бронная, 17",66.0,1150000,17424.2,Тверская,5.0


In [23]:
!pip install -q dash

from dash import Dash, dcc, html, Input, Output, dash_table
import plotly.express as px
import threading
from google.colab.output import eval_js

app = Dash(__name__)

# стили карточки — используем везде где нужен белый блок с тенью
card = {
    "backgroundColor": "#ffffff",
    "borderRadius": "12px",
    "padding": "20px",
    "boxShadow": "0 1px 3px rgba(0,0,0,0.08)",
    "marginBottom": "16px",
}

app.layout = html.Div(
    style={"backgroundColor": "#f8fafc", "minHeight": "100vh",
           "fontFamily": "Arial, sans-serif", "padding": "24px"},
    children=[

        # заголовок
        html.Div([
            html.H1("DarkSpot — аренда коммерции (ЦИАН)",
                    style={"margin": 0, "color": "#1e293b", "fontSize": "24px", "fontWeight": 700}),
            html.P("Москва, статический парсинг",
                   style={"margin": "4px 0 0", "color": "#64748b", "fontSize": "14px"}),
        ], style={**card, "marginBottom": "20px"}),

        # фильтры
        html.Div([
            html.Div([
                html.Label("тип объекта", style={"fontWeight": 600, "fontSize": "13px"}),
                dcc.Dropdown(
                    id="filter-type",
                    options=[{"label": "все типы", "value": "ALL"}] +
                            [{"label": t, "value": t} for t in sorted(df["object_type"].dropna().unique())],
                    value="ALL", clearable=False, style={"marginTop": "6px"},
                ),
            ], style={"flex": 1, "marginRight": "16px"}),

            html.Div([
                html.Label("округ", style={"fontWeight": 600, "fontSize": "13px"}),
                dcc.Dropdown(
                    id="filter-okrug",
                    options=[{"label": "все округа", "value": "ALL"}] +
                            [{"label": o, "value": o} for o in sorted(df["okrug"].dropna().unique())],
                    value="ALL", clearable=False, style={"marginTop": "6px"},
                ),
            ], style={"flex": 1, "marginRight": "16px"}),

            html.Div([
                html.Label("ценовая категория", style={"fontWeight": 600, "fontSize": "13px"}),
                dcc.Dropdown(
                    id="filter-tier",
                    options=[{"label": "все категории", "value": "ALL"}] +
                            [{"label": str(t), "value": str(t)} for t in df["price_tier_per_meter"].cat.categories],
                    value="ALL", clearable=False, style={"marginTop": "6px"},
                ),
            ], style={"flex": 1}),
        ], style={**card, "display": "flex", "alignItems": "flex-end"}),

        # kpi-карточки
        html.Div(id="kpi-row", style={"display": "grid", "gridTemplateColumns": "repeat(4, 1fr)",
                                       "gap": "16px", "marginBottom": "16px"}),

        # графики
        html.Div([
            html.Div(dcc.Graph(id="hist-price"), style={**card, "flex": 1}),
            html.Div(dcc.Graph(id="pie-type"),   style={**card, "flex": 1}),
        ], style={"display": "flex", "gap": "16px"}),

        html.Div([
            html.Div(dcc.Graph(id="bar-okrug"), style={**card, "flex": 1}),
            html.Div(dcc.Graph(id="bar-metro"), style={**card, "flex": 1}),
        ], style={"display": "flex", "gap": "16px"}),

        html.Div(dcc.Graph(id="scatter-area"), style=card),
        html.Div(dcc.Graph(id="box-type"),     style=card),

        # топ-10
        html.Div([
            html.H3("топ-10 по цене за м²", style={"marginTop": 0, "fontWeight": 600}),
            html.Div(id="top-table"),
        ], style=card),
    ]
)


# единый callback — пересчитывает все графики при изменении любого фильтра
@app.callback(
    Output("kpi-row",      "children"),
    Output("hist-price",   "figure"),
    Output("pie-type",     "figure"),
    Output("bar-okrug",    "figure"),
    Output("bar-metro",    "figure"),
    Output("scatter-area", "figure"),
    Output("box-type",     "figure"),
    Output("top-table",    "children"),
    Input("filter-type",   "value"),
    Input("filter-okrug",  "value"),
    Input("filter-tier",   "value"),
)
def update_all(f_type, f_okrug, f_tier):
    dff = df.copy()
    if f_type  != "ALL": dff = dff[dff["object_type"] == f_type]
    if f_okrug != "ALL": dff = dff[dff["okrug"]       == f_okrug]
    if f_tier  != "ALL": dff = dff[dff["price_tier_per_meter"].astype(str) == f_tier]

    n = len(dff)

    # маленькая карточка с одной метрикой
    def kpi_card(label, value, unit=""):
        return html.Div([
            html.P(label, style={"margin": 0, "fontSize": "12px", "color": "#64748b"}),
            html.P(f"{value} {unit}", style={"margin": "4px 0 0", "fontSize": "22px",
                                             "fontWeight": 700, "color": "#1d4ed8"}),
        ], style={**card, "marginBottom": 0})

    kpis = [
        kpi_card("объявлений",       f"{n:,}"),
        kpi_card("медиана цены",     f"{dff['price_sqm_month'].median():.0f}" if n else "-", "руб/м²/мес"),
        kpi_card("медиана площади",  f"{dff['area_sqm'].median():.0f}"        if n else "-", "м²"),
        kpi_card("медиана до метро", f"{dff['metro_walk_min'].median():.0f}"  if dff["metro_walk_min"].notna().any() else "-", "мин"),
    ]

    lay = dict(plot_bgcolor="white", paper_bgcolor="white",
               margin=dict(t=45, b=30, l=40, r=20), height=360)

    fig_hist = px.histogram(dff, x="price_sqm_month", nbins=40,
                            title="распределение цены (руб/м²/мес)",
                            color_discrete_sequence=["#1d4ed8"])
    if n:
        fig_hist.add_vline(x=dff["price_sqm_month"].median(), line_dash="dash", line_color="crimson",
                           annotation_text=f"медиана: {dff['price_sqm_month'].median():.0f}",
                           annotation_position="top right")
    fig_hist.update_layout(**lay)

    type_cnt = dff["object_type"].value_counts().reset_index()
    type_cnt.columns = ["object_type", "count"]
    fig_pie = px.pie(type_cnt, names="object_type", values="count",
                     title="структура по типам объектов",
                     hole=0.42, color_discrete_sequence=px.colors.qualitative.Pastel)
    fig_pie.update_layout(**lay)

    bar_df = dff.groupby("okrug")["price_sqm_month"].median().sort_values(ascending=False).reset_index()
    fig_bar = px.bar(bar_df, x="okrug", y="price_sqm_month",
                     title="медиана цены по округам",
                     color="price_sqm_month", color_continuous_scale="Blues")
    fig_bar.update_layout(**lay, coloraxis_showscale=False)

    metro_df = (dff[dff["metro_walk_min"] > 0]
                .groupby("metro_tier", observed=True)["price_sqm_month"]
                .median().reset_index())
    fig_metro = px.bar(metro_df, x="metro_tier", y="price_sqm_month",
                       title="цена vs расстояние до метро",
                       color="price_sqm_month", color_continuous_scale="Oranges", text_auto=".0f")
    fig_metro.update_layout(**lay, coloraxis_showscale=False)

    fig_sc = px.scatter(dff, x="area_sqm", y="price_sqm_month",
                        color="object_type", size="price_month", size_max=18, opacity=0.7,
                        hover_data=["okrug", "district", "metro_station", "metro_walk_min", "address"],
                        title="площадь vs цена за м²",
                        color_discrete_sequence=px.colors.qualitative.Set2)
    fig_sc.update_layout(**lay, height=430)

    fig_box = px.box(dff, x="object_type", y="price_sqm_month", color="object_type",
                     title="разброс цен по типам объектов",
                     color_discrete_sequence=px.colors.qualitative.Set2)
    fig_box.update_layout(**lay, showlegend=False)

    top = (dff.sort_values("price_sqm_month").head(10)
           [["url", "object_type", "okrug", "area_sqm", "price_month", "price_sqm_month",
             "metro_station", "metro_walk_min"]]
           .reset_index(drop=True))
    top.index += 1
    top["price_month"]     = top["price_month"].map("{:,.0f} руб".format)
    top["price_sqm_month"] = top["price_sqm_month"].map("{:,.0f} руб/м²".format)
    top["area_sqm"]        = top["area_sqm"].map("{:.0f} м²".format)

    table = dash_table.DataTable(
        data=top.to_dict("records"),
        columns=[{"name": c, "id": c} for c in top.columns],
        style_cell={"fontSize": "13px", "padding": "8px 12px",
                    "textOverflow": "ellipsis", "maxWidth": "220px"},
        style_header={"fontWeight": 700, "backgroundColor": "#f1f5f9"},
        style_data_conditional=[{"if": {"row_index": "odd"}, "backgroundColor": "#f8fafc"}],
        style_table={"overflowX": "auto"},
    )

    return kpis, fig_hist, fig_pie, fig_bar, fig_metro, fig_sc, fig_box, table


# запускаем сервер в фоне и печатаем ссылку
threading.Thread(target=lambda: app.run(debug=False, port=8050), daemon=True).start()
print("дэшборд:", eval_js("google.colab.kernel.proxyPort(8050)"))

Dash is running on http://127.0.0.1:8050/



INFO:dash.dash:Dash is running on http://127.0.0.1:8050/



 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 8050 is in use by another program. Either identify and stop that program, or start the server with a different port.


дэшборд: https://8050-gpu-t4-s-kkb-usw4a2-cceohvsk9ns2-a.us-west4-2.prod.colab.dev


# Выводы

Собрали данные по коммерческой аренде с ЦИАНа — офисы, торговые площади, свободное назначение, склады. Поля: цена, площадь, тип объекта, округ, район, метро, этаж, класс здания, ремонт.

## Что нашли

**Округа.** ЦАО дороже всех — ставки в 1.5-2 раза выше периферии. ЗАО и ЮЗАО — второй эшелон, хорошая доступность при умеренной цене. ЮВАО и ЮАО — самые дешевые, подходят для ПВЗ и складов.

**Метро.** Близость к метро заметно влияет на цену. Для кофеен и торговли — критично, для складов и бэк-офисов — не так важно, можно сэкономить.

**Структура рынка.** Большая часть предложения — офисы и торговые площади. Эконом (<5000 руб/м²) преобладает в складах и офисах, люкс (>40 000 руб/м²) сосредоточен в ЦАО.

**Площадь и цена.** Чем больше помещение — тем ниже ставка за м². Малые форматы 20-80 м² самые дорогие удельно, но и самые ликвидные.

## Рекомендации

| Формат | Округ | Площадь |
|---|---|---|
| Кофейня | ЦАО, ЗАО | 30-60 м² |
| Салон красоты | ЮЗАО, ЗАО | 40-80 м² |
| ПВЗ | ЮВАО, ЮАО | 20-40 м² |
| Офис | САО, СВАО | 50-150 м² |

## Связь с блоком ЦА

Данные по аренде + данные по целевой аудитории дают комплексный сигнал: высокая плотность платёжеспособной аудитории и разумная ставка вместе — приоритетная локация. Смотреть надо на оба параметра сразу.